In [ ]:
!pip install qiskit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 95.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 79.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [qiskit]


In [ ]:
from qiskit import QuantumCircuit
from qiskit.compiler import transpile
from qiskit.dagcircuit import DAGCircuit
from qiskit.converters import circuit_to_dag, dag_to_circuit
from qiskit import qasm2


# 1. Создаем тестовую "грязную" схему, прямо как в соревновании
# Представим, что мы прочитали этот текст из файла OpenQASM
qasm_code = """
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];

t q[0];
t q[0];    // Биграмма из двух T: должна превратиться в S

t q[1];
tdg q[1];  // Биграмма T и T-dagger: должны полностью уничтожиться
"""

# Загружаем этот текст в объект квантовой схемы Python
circuit = QuantumCircuit.from_qasm_str(qasm_code)
print("--- ИСХОДНАЯ СХЕМА (До оптимизации) ---")
print(qasm2.dumps(circuit))


# 2. Переводим схему в Направленный Ациклический Граф (DAG)
# Это нужно, чтобы вентили распределились по "дорожкам" кубитов
dag = circuit_to_dag(circuit)

# 3. Логика оптимизатора: проходим по графу и ищем биграммы
def optimize_t_gates(dag: DAGCircuit) -> DAGCircuit:
    # Метод nodes_on_wire_and_nodes_on_edges позволяет идти строго по линии каждого кубита
    for wire in dag.wires:
        # Получаем последовательность всех вентилей на конкретном кубите
        nodes = list(dag.nodes_on_wire(wire))

        i = 0
        while i < len(nodes) - 1:
            node1 = nodes[i]
            node2 = nodes[i+1]

            # Проверяем, являются ли два соседних узла квантовыми гейтами
            if node1.type == 'op' and node2.type == 'op':

                # Сценарий А: Нашли [T] + [T] -> заменяем на [S]
                if node1.name == 't' and node2.name == 't':
                    # Удаляем старые узлы T из графа
                    dag.remove_op_node(node1)
                    dag.remove_op_node(node2)
                    # На их место вставляем один дешевый гейт S
                    # (В реальном коде турнира здесь также прописывается точное место вставки)

                # Сценарий Б: Нашли [T] + [T-dagger] -> полностью удаляем
                elif (node1.name == 't' and node2.name == 'tdg') or (node1.name == 'tdg' and node2.name == 't'):
                    dag.remove_op_node(node1)
                    dag.remove_op_node(node2)

            i += 1
    return dag

# Применяем наш алгоритм (в Qiskit есть встроенные аналоги, но этот написан вручную для наглядности)
# Для полной и глубокой оптимизации в Qiskit используют Transpiler Passes:
optimized_circuit = transpile(circuit, optimization_level=1) # Level 1 как раз делает слияние фаз

print("\n--- ОПТИМИЗИРОВАННАЯ СХЕМА ---")
print(qasm2.dumps(optimized_circuit))




--- ИСХОДНАЯ СХЕМА (До оптимизации) ---
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
t q[0];
t q[0];
t q[1];
tdg q[1];

--- ОПТИМИЗИРОВАННАЯ СХЕМА ---
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
u1(pi/2) q[0];


In [ ]:
!pip install matplotlib pydot graphviz


In [ ]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_dag
from qiskit.visualization import dag_drawer
import matplotlib.pyplot as plt

# 1. Создаем схему с двухкубитным вентилем CNOT (cx)
# Кубит 0 будет ведущим (Control), Кубит 1 - ведомым (Target)
qasm_with_cnot = """
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];

t q[0];            // Гейт на ведущем кубите
cx q[0], q[1];     // CNOT: q[0] управляет, q[1] изменяется
t q[1];            // Гейт на ведомом кубите
"""


circuit = QuantumCircuit.from_qasm_str(qasm_with_cnot)

# 2. Превращаем схему в граф (DAG)
dag = circuit_to_dag(circuit)

# 3. Визуализируем граф зависимостей
# Эта функция сгенерирует изображение, где узлы — это гейты,
# а стрелки показывают, как данные текут по ведущим и ведомым ветвям.
dag_drawer(dag, filename="quantum_dag.png")

print("Граф схемы успешно сохранен в файл 'quantum_dag.png'!")


Граф схемы успешно сохранен в файл 'quantum_dag.png'!


In [ ]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_dag

# 1. Загружаем схему
qasm_with_cnot = """
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];

t q[0];
cx q[0], q[1];
t q[1];
"""

circuit = QuantumCircuit.from_qasm_str(qasm_with_cnot)
dag = circuit_to_dag(circuit)

# 2. Новая функция анализа с поддержкой Qiskit 1.x
def analyze_cnot_connections(dag_circuit):
    print("=== АНАЛИЗ ДВУХКУБИТНЫХ СВЯЗЕЙ В ГРАФЕ ===\n")

    # Создаем карту индексов кубитов прямо из графа
    qubit_map = {qubit: index for index, qubit in enumerate(dag_circuit.qubits)}

    for node in dag_circuit.op_nodes():
        if node.name == 'cx':
            # Извлекаем объекты кубитов
            control_qubit = node.qargs[0]
            target_qubit = node.qargs[1]

            # Находим их порядковые номера через созданную карту
            c_index = qubit_map[control_qubit]
            t_index = qubit_map[target_qubit]

            c_name = f"q[{c_index}]"
            t_name = f"q[{t_index}]"

            print(f" Найдена двухкубитная связь: вентиль [{node.name.upper()}]")
            print(f"   ├─ ВЕДУЩАЯ ветвь (Control): {c_name}")
            print(f"   └─ ВЕДОМАЯ ветвь (Target) : {t_name}")
            print("-" * 40)

# Запускаем анализ
analyze_cnot_connections(dag)


=== АНАЛИЗ ДВУХКУБИТНЫХ СВЯЗЕЙ В ГРАФЕ ===

 Найдена двухкубитная связь: вентиль [CX]
   ├─ ВЕДУЩАЯ ветвь (Control): q[0]
   └─ ВЕДОМАЯ ветвь (Target) : q[1]
----------------------------------------


In [ ]:
from qiskit import QuantumCircuit, qasm2
from qiskit.converters import circuit_to_dag, dag_to_circuit
from qiskit.circuit.library import TGate
# Импортируем класс операционного узла для проверки типа
from qiskit.dagcircuit import DAGOpNode

qasm_before = """
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];

t q[0];
cx q[0],q[1];
"""

circuit = QuantumCircuit.from_qasm_str(qasm_before)
dag = circuit_to_dag(circuit)

print("--- СХЕМА ДО ПРОТАСКИВАНИЯ ---")
print(qasm2.dumps(dag_to_circuit(dag)))

def commute_t_forward(dag_circuit):
    for cx_node in list(dag_circuit.op_nodes()):
        if cx_node.name == 'cx':
            control_qubit = cx_node.qargs[0] # Берем ведущий кубит

            for pred in dag_circuit.predecessors(cx_node):
                # Изменение здесь: вместо .type == 'op' проверяем класс узла через isinstance
                if isinstance(pred, DAGOpNode) and control_qubit in pred.qargs and pred.name == 't':

                    print(f"\n[Успех] Найдена возможность комммутации!")
                    print(f"Протаскиваем гейт [{pred.name.upper()}] вперед через ведущую линию CNOT...")

                    # Удаляем старый узел T
                    dag_circuit.remove_op_node(pred)

                    # Вставляем новый T-гейт ПОСЛЕ cx_node на ведущий кубит.
                    # apply_operation_back добавляет операцию в самый конец текущего пути кубита
                    dag_circuit.apply_operation_back(TGate(), qargs=[control_qubit])

                    print("Перестановка успешно выполнена!")
                    return dag_circuit

    return dag_circuit

optimized_dag = commute_t_forward(dag)
optimized_circuit = dag_to_circuit(optimized_dag)

print("\n--- СХЕМА ПОСЛЕ АВТОМАТИЧЕСКОГО ПРОТАСКИВАНИЯ ---")
print(qasm2.dumps(optimized_circuit))


--- СХЕМА ДО ПРОТАСКИВАНИЯ ---
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
t q[0];
cx q[0],q[1];

[Успех] Найдена возможность комммутации!
Протаскиваем гейт [T] вперед через ведущую линию CNOT...
Перестановка успешно выполнена!

--- СХЕМА ПОСЛЕ АВТОМАТИЧЕСКОГО ПРОТАСКИВАНИЯ ---
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
cx q[0],q[1];
t q[0];


In [ ]:
from qiskit import QuantumCircuit, qasm2
from qiskit.converters import circuit_to_dag, dag_to_circuit
from qiskit.circuit.library import TGate, SGate
from qiskit.dagcircuit import DAGOpNode

# 1. Более сложная "грязная" схема: T-гейты разделены CNOT, но стоят на одной ведущей линии!
qasm_complex = """
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];

t q[0];         // Первый дорогой T-гейт (до CNOT)
cx q[0],q[1];   // CNOT связь
t q[0];         // Второй дорогой T-гейт (после CNOT)
"""

circuit = QuantumCircuit.from_qasm_str(qasm_complex)
dag = circuit_to_dag(circuit)

print("--- ИСХОДНАЯ СХЕМА ---")
print(qasm2.dumps(dag_to_circuit(dag)))

def full_pipeline_optimizer(dag_circuit):
    # --- ШАГ 1: Протаскивание (Коммутация) ---
    for cx_node in list(dag_circuit.op_nodes()):
        if cx_node.name == 'cx':
            control_qubit = cx_node.qargs[0]

            for pred in dag_circuit.predecessors(cx_node):
                if isinstance(pred, DAGOpNode) and control_qubit in pred.qargs and pred.name == 't':
                    print("\n[Оптимизатор]: Найдено скрытое T-звено. Протаскиваем сквозь CNOT...")
                    dag_circuit.remove_op_node(pred)
                    dag_circuit.apply_operation_back(TGate(), qargs=[control_qubit])
                    break # Успешно передвинули, выходим из внутреннего цикла

    # --- ШАГ 2: Поиск n-грамм и схлопывание фаз ---
    # Перестраиваем последовательность узлов после изменения графа
    for wire in dag_circuit.wires:
        nodes = list(dag_circuit.nodes_on_wire(wire))
        i = 0
        while i < len(nodes) - 1:
            node1 = nodes[i]
            node2 = nodes[i+1]

            if isinstance(node1, DAGOpNode) and isinstance(node2, DAGOpNode):
                # Если два T-гейта оказались рядом после переноса:
                if node1.name == 't' and node2.name == 't':
                    print("[Оптимизатор]: Бум! Обнаружена биграмма [T] + [T]. Схлопываем в гейт [S]...")
                    dag_circuit.remove_op_node(node1)
                    dag_circuit.remove_op_node(node2)
                    dag_circuit.apply_operation_back(SGate(), qargs=[wire])
                    break
            i += 1

    return dag_circuit

# Запускаем полный пайплайн
optimized_dag = full_pipeline_optimizer(dag)
optimized_circuit = dag_to_circuit(optimized_dag)

print("\n--- ФИНАЛЬНАЯ ОПТИМИЗИРОВАННАЯ СХЕМА ---")
print(qasm2.dumps(optimized_circuit))


--- ИСХОДНАЯ СХЕМА ---
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
t q[0];
cx q[0],q[1];
t q[0];

[Оптимизатор]: Найдено скрытое T-звено. Протаскиваем сквозь CNOT...
[Оптимизатор]: Бум! Обнаружена биграмма [T] + [T]. Схлопываем в гейт [S]...

--- ФИНАЛЬНАЯ ОПТИМИЗИРОВАННАЯ СХЕМА ---
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
cx q[0],q[1];
s q[0];


In [ ]:
from qiskit.circuit.library import TGate
from qiskit.dagcircuit import DAGOpNode

def commute_t_backward_surgery(dag_circuit):
    # Превращаем в список, так как граф нельзя модифицировать во время итерации
    for cx_node in list(dag_circuit.op_nodes()):
        if cx_node.name == 'cx':
            # В Qiskit 1.x qargs возвращает кубиты. Ведущий (Control) — первый в списке [0]
            control_qubit = cx_node.qargs[0]

            # Ищем последователей CNOT
            for succ in list(dag_circuit.successors(cx_node)):
                # Проверяем: это операция, она стоит на ведущем кубите и это гейт 't'
                if isinstance(succ, DAGOpNode) and control_qubit in succ.qargs and succ.name == 't':

                    print(f"\n[Успех] Найдена возможность обратной коммутации!")
                    print(f"Ротируем гейт [T] НАЗАД через ведущую линию CNOT...")

                    # ИСПРАВЛЕНИЕ ПО СТАТЬЕ:
                    # 1. Удаляем Т-гейт с его текущей позиции (после CNOT)
                    dag_circuit.remove_op_node(succ)

                    # 2. Хирургически вставляем Т-гейт СТРОГО ПЕРЕД узлом CNOT
                    # Метод insert_op_before_node автоматически перестроит указатели рёбер графа
                    dag_circuit.insert_op_before_node(cx_node, TGate(), qargs=[control_qubit])

                    print("Локальная хирургия графа успешно выполнена!")
                    return dag_circuit # Возвращаем изменённый граф

    return dag_circuit


In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import TGate, HGate
from qiskit.converters import circuit_to_dag, dag_to_circuit
from qiskit.dagcircuit import DAGOpNode
from qiskit.quantum_info import Operator

# =====================================================================
# 1. СОЗДАНИЕ ТЕСТОВОЙ СХЕМЫ С "ПОМЕХОЙ"
# =====================================================================
# Создаем схему, где гейт 'h' (Адамар) является непробиваемым барьером.
# T-гейт на q[0] после CNOT можно подвинуть НАЗАД (коммутирует с контрольной линией).
# Но он НЕ ДОЛЖЕН перепрыгнуть гейт H в самом начале, иначе схема сломается!

test_circuit = QuantumCircuit(2)
test_circuit.h(0)           # <-- Помеха в начале (T с ней не коммутирует!)
test_circuit.cx(0, 1)       # CNOT
test_circuit.t(0)           # <-- Этот гейт мы хотим аккуратно сдвинуть НАЗАД перед CNOT

print("--- ИСХОДНАЯ ТЕСТОВАЯ СХЕМА ---")
print(test_circuit)

# Сохраняем эталонную матрицу исходной схемы для проверки эквивалентности
original_operator = Operator(test_circuit)


# =====================================================================
# 2. РЕАЛИЗАЦИЯ ХИРУРГИИ ГРАФА ПО МЕТОДОЛОГИИ СТАТЬИ
# =====================================================================

def commute_t_backward_surgery(circuit):
    """Безопасно двигает T-гейт НАЗАД сквозь контрольную линию CNOT на один шаг"""
    dag = circuit_to_dag(circuit)

    for cx_node in list(dag.op_nodes()):
        if cx_node.name == 'cx':
            control_qubit = cx_node.qargs[0] # Ведущий кубит

            # Проверяем последователей (гейты ПОСЛЕ CNOT)
            for succ in list(dag.successors(cx_node)):
                if isinstance(succ, DAGOpNode) and control_qubit in succ.qargs and succ.name == 't':
                    print("\n[Хирургия Назад] Найдена возможность обратной коммутации.")

                    # Перерезаем связи старого узла и удаляем его
                    dag.remove_op_node(succ)

                    # Сшиваем новый T-гейт СТРОГО ПЕРЕД этим CNOT
                    dag.insert_op_before_node(cx_node, TGate(), qargs=[control_qubit])

                    print("-> Гейт [T] аккуратно протиснут НАЗАД, не перепрыгивая барьеры в начале.")
                    return dag_to_circuit(dag)
    return circuit


def commute_t_forward_surgery(circuit):
    """Безопасно двигает T-гейт ВПЕРЕД сквозь контрольную линию CNOT на один шаг"""
    dag = circuit_to_dag(circuit)

    for cx_node in list(dag.op_nodes()):
        if cx_node.name == 'cx':
            control_qubit = cx_node.qargs[0] # Ведущий кубит

            # Проверяем предшественников (гейты ПЕРЕД CNOT)
            for pred in list(dag.predecessors(cx_node)):
                if isinstance(pred, DAGOpNode) and control_qubit in pred.qargs and pred.name == 't':
                    print("\n[Хирургия Вперед] Найдена возможность прямой коммутации.")

                    # Удаляем T-гейт с позиции перед CNOT
                    dag.remove_op_node(pred)

                    # Вставляем новый T-гейт СТРОГО ПОСЛЕ этого CNOT
                    # (Для этого используем insert_op_after_node)
                    dag.insert_op_after_node(cx_node, TGate(), qargs=[control_qubit])

                    print("-> Гейт [T] аккуратно протиснут ВПЕРЕД сквозь CNOT.")
                    return dag_to_circuit(dag)
    return circuit


# =====================================================================
# 3. ЗАПУСК И ПРОВЕРКА ЭКВИВАЛЕНТНОСТИ
# =====================================================================

# Шаг А: Применяем движение НАЗАД
from qiskit.dagcircuit import DAGCircuit

# ... (ваш код итерации по cx_node)

# Создаем мини-DAG для замены
mini_dag = DAGCircuit()
for q in cx_node.qargs:
    mini_dag.add_qubits([q])  # Регистрируем кубиты в мини-графе

# Добавляем сначала T-гейт, затем CX-гейт (двигаем T назад перед CX)
mini_dag.apply_operation_back(TGate(), qargs=[control_qubit])
mini_dag.apply_operation_back(cx_node.op, qargs=cx_node.qargs)

# Заменяем старый cx_node целиком на наш новый мини-граф
dag.substitute_node_with_dag(cx_node, mini_dag)

circuit_backward = commute_t_backward_surgery(test_circuit)
print("\n--- СХЕМА ПОСЛЕ КОММУТАЦИИ НАЗАД ---")
print(circuit_backward)

# Шаг Б: Проверяем, не сломалась ли математика
backward_operator = Operator(circuit_backward)
is_backward_valid = original_operator.equiv(backward_operator)
print(f"Математически эквивалентна исходной? {'🟢 ДА (Успех хирургии)' if is_backward_valid else '🔴 НЕТ (Схема сломана)'}")


# Шаг В: Возвращаем гейт обратно, применяя движение ВПЕРЕД
circuit_forward = commute_t_forward_surgery(circuit_backward)
print("\n--- СХЕМА ПОСЛЕ КОММУТАЦИИ ВПЕРЕД ---")
print(circuit_forward)

# Шаг Г: Снова проверяем эквивалентность
forward_operator = Operator(circuit_forward)
is_forward_valid = original_operator.equiv(forward_operator)
print(f"Математически эквивалентна исходной? {'🟢 ДА (Успех хирургии)' if is_forward_valid else '🔴 НЕТ (Схема сломана)'}")


--- ИСХОДНАЯ ТЕСТОВАЯ СХЕМА ---
     ┌───┐     ┌───┐
q_0: ┤ H ├──■──┤ T ├
     └───┘┌─┴─┐└───┘
q_1: ─────┤ X ├─────
          └───┘     


NameError: name 'cx_node' is not defined

In [ ]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_dag, dag_to_circuit
from qiskit.dagcircuit import DAGCircuit
from qiskit.circuit.library import CXGate, TGate
from qiskit.quantum_info import Operator

# Создаем тестовую схему (пример, где T-гейт стоит после CX на control-кубите)
test_circuit = QuantumCircuit(2)
test_circuit.cx(0, 1)
test_circuit.t(0)

def commute_t_backward_surgery(circuit: QuantumCircuit) -> QuantumCircuit:
    """ Перемещает T-гейт назад (налево), проталкивая его ПЕРЕД гейтом CX """
    dag = circuit_to_dag(circuit)

    # Итерируемся по CX-узлам
    for cx_node in dag.op_nodes(CXGate):
        # Первый кубит в гейте CX — это всегда control (управляющий)
        control_qubit = cx_node.qargs[0]

        # Используем op_successors, чтобы гарантированно получать только гейты
        for succ in list(dag.op_successors(cx_node)):
            # Если следующий гейт — TGate и он висит именно на нашем control-кубите
            if isinstance(succ.op, TGate) and control_qubit in succ.qargs:

                # Создаем мини-DAG для замены
                mini_dag = DAGCircuit()
                for q in cx_node.qargs:
                    mini_dag.add_qubits([q])

                # Сначала добавляем T, затем сам CX (протолкнули назад)
                # Передаем обычный плоский список [control_qubit]
                mini_dag.apply_operation_back(TGate(), qargs=[control_qubit])
                mini_dag.apply_operation_back(cx_node.op, qargs=cx_node.qargs)

                # Заменяем старый узел и удаляем исходный T
                dag.substitute_node_with_dag(cx_node, mini_dag)
                dag.remove_op_node(succ)
                break

    return dag_to_circuit(dag)


def commute_t_forward_surgery(circuit: QuantumCircuit) -> QuantumCircuit:
    """ Перемещает T-гейт вперед (направо), проталкивая его ЗА гейт CX """
    dag = circuit_to_dag(circuit)

    for cx_node in dag.op_nodes(CXGate):
        # Явно берем именно первый кубит (control) из пары кубитов CNOT
        control_qubit = cx_node.qargs[0]

        # Используем op_predecessors, чтобы отсечь концевые узлы
        for pred in list(dag.op_predecessors(cx_node)):
            # Если перед CX стоял TGate на том же самом control-кубите
            if isinstance(pred.op, TGate) and control_qubit in pred.qargs:

                # Создаем мини-DAG для замены
                mini_dag = DAGCircuit()
                for q in cx_node.qargs:
                    mini_dag.add_qubits([q])

                # Сначала добавляем CX, затем T (протолкнули вперед)
                mini_dag.apply_operation_back(cx_node.op, qargs=cx_node.qargs)
                # Передаем чистый плоский список с одним кубитом
                mini_dag.apply_operation_back(TGate(), qargs=[control_qubit])

                # Заменяем старый узел и удаляем исходный T
                dag.substitute_node_with_dag(cx_node, mini_dag)
                dag.remove_op_node(pred)
                break

    return dag_to_circuit(dag)


# --- ТЕСТИРОВАНИЕ КОММУТАЦИИ НАЗАД ---
circuit_backward = commute_t_backward_surgery(test_circuit)
print("\n--- СХЕМА ПОСЛЕ КОММУТАЦИИ НАЗАД ---")
print(circuit_backward)

# Шаг Б: Проверяем, не сломалась ли математика
backward_operator = Operator(circuit_backward)
is_backward_valid = original_operator.equiv(backward_operator)
print(f"Математически эквивалентна исходной? { '🟢 ДА (Успех хирургии)' if is_backward_valid else '🔴 НЕТ (Схема сломана)' }")

# --- ТЕСТИРОВАНИЕ КОММУТАЦИИ ВПЕРЕД ---
# Шаг В: Возвращаем гейт обратно, применяя движение ВПЕРЕД
circuit_forward = commute_t_forward_surgery(circuit_backward)
print("\n--- СХЕМА ПОСЛЕ КОММУТАЦИИ ВПЕРЕД ---")
print(circuit_forward)

# Шаг Г: Снова проверяем эквивалентность
forward_operator = Operator(circuit_forward)
is_forward_valid = original_operator.equiv(forward_operator)
print(f"Математически эквивалентна исходной? { '🟢 ДА (Успех хирургии)' if is_forward_valid else '🔴 НЕТ (Схема сломана)' }")



--- СХЕМА ПОСЛЕ КОММУТАЦИИ НАЗАД ---
     ┌───┐     
q_0: ┤ T ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
          └───┘
Математически эквивалентна исходной? 🟢 ДА (Успех хирургии)

--- СХЕМА ПОСЛЕ КОММУТАЦИИ ВПЕРЕД ---
          ┌───┐
q_0: ──■──┤ T ├
     ┌─┴─┐└───┘
q_1: ┤ X ├─────
     └───┘     
Математически эквивалентна исходной? 🟢 ДА (Успех хирургии)


In [ ]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_dag, dag_to_circuit

class QuantumLumberjackOptimizer:
    """
    Оптимизатор квантовых цепей на базе анализа DAG и N-грамм.
    Специализируется на Clifford+T базисе для турнира Kaggle.
    """
    def __init__(self, circuit: QuantumCircuit):
        # Делаем копию исходной схемы, чтобы случайно не испортить оригинал
        self.circuit = circuit.copy()
        # Переводим копию схемы в граф (DAG) для хирургии
        self.dag = circuit_to_dag(self.circuit)

    def prune_dead_cnots(self):  # <- Исправлена опечатка (было pruno)
        """
        БЛОК 1: Разведка и расчистка.
        Находит гейты CNOT, перед которыми гарантированно стоит ноль,
        и хирургически удаляет их из графа.
        """
        print("[Робот-Разведчик] Ищу холостые CNOT...")
        # TODO: Напишем этот код вместе через 6-7 часов
        pass

    def optimize_t_ngrams(self):
        """
        БЛОК 2: Хирургия и Тетрис.
        Запускает скользящее окно по нитям графа, двигает T-гейты
        вперед-назад и схлопывает биграммы/триграммы.
        """
        print("[Робот-Хирург] Сканирую провода на наличие N-грамм...")
        # TODO: Реализуем скользящее окно и правила коммутации
        pass

    def compile(self) -> QuantumCircuit:
        """
        Главный пайплайн компиляции.
        Запускает блоки оптимизации по очереди и возвращает сжатую схему.
        """
        print("\n--- ЗАПУСК КВАНТОВОГО СЖАТИЯ ---")

        # 1. Сначала убираем плотины (CNOT)
        self.prune_dead_cnots()

        # 2. Потом гоняем T-гейты по свободным проводам
        self.optimize_t_ngrams()

        # Превращаем измененный граф обратно в стандартную схему Qiskit
        return dag_to_circuit(self.dag)


In [ ]:
# Создаем грязную «нескомпилированную» тестовую схему
qc = QuantumCircuit(2)

# Линия 0 пустая, значит этот CNOT холостой:
qc.cx(0, 1)

# Ставим два T-гейта подряд на кубит 0 (они должны превратиться в S)
qc.t(0)
qc.t(0)

print("--- ИСХОДНАЯ СХЕМА ---")
print(qc)

# Запускаем нашего Лесоруба-Оптимизатора
optimizer = QuantumLumberjackOptimizer(qc)
compressed_qc = optimizer.compile()

print("\n--- СЖАТАЯ СХЕМА ---")
print(compressed_qc)


--- ИСХОДНАЯ СХЕМА ---
          ┌───┐┌───┐
q_0: ──■──┤ T ├┤ T ├
     ┌─┴─┐└───┘└───┘
q_1: ┤ X ├──────────
     └───┘          

--- ЗАПУСК КВАНТОВОГО СЖАТИЯ ---
[Робот-Разведчик] Ищу холостые CNOT...
  -> Найдена плотина! Удаляю холостой CNOT на кубите <Qubit register=(2, "q"), index=0>
[Робут-Хирург] Сканирую провода на наличие N-грамм...
  -> Найдена биграмма! Схлопываю T + T в S на кубите (<Qubit register=(2, "q"), index=0>,)

--- СЖАТАЯ СХЕМА ---
     ┌───┐
q_0: ┤ S ├
     └───┘
q_1: ─────
          


In [ ]:
print("=== ТЕСТ 2: ТОЛЬКО СХЛОПЫВАНИЕ T-ГЕЙТОВ ===")
t_only_qc = QuantumCircuit(*large_qc.qregs)

for instruction in large_qc.data:
    gate = instruction.operation
    qubit_indices = [large_qc.find_bit(q).index for q in instruction.qubits]

    if isinstance(gate, TGate):
        q_idx = qubit_indices[0]

        # Заглядываем в самый конец текущей сборки
        if len(t_only_qc.data) > 0:
            last_instruction = t_only_qc.data[-1]
            last_gate = last_instruction.operation
            last_qubits = [t_only_qc.find_bit(q).index for q in last_instruction.qubits]

            # Схлопываем, ТОЛЬКО если предыдущим гейтом во всей схеме был T на этом же кубите
            if isinstance(last_gate, TGate) and last_qubits == [q_idx]:
                print(f"  -> Схлопываю T + T в S на кубите index={q_idx}")
                t_only_qc.data.pop()
                t_only_qc.append(SGate(), instruction.qubits)
                continue

    t_only_qc.append(gate, instruction.qubits)

# Валидация
t_only_op = Operator(t_only_qc)
print(f"Математика Т-гейтов исправна? { '🟢 ДА (Хирург чист)' if original_op.equiv(t_only_op) else '🔴 НЕТ (Хирург ломает)' }")


=== ТЕСТ 2: ТОЛЬКО СХЛОПЫВАНИЕ T-ГЕЙТОВ ===
  -> Схлопываю T + T в S на кубите index=2
  -> Схлопываю T + T в S на кубите index=1
  -> Схлопываю T + T в S на кубите index=2
  -> Схлопываю T + T в S на кубите index=0
  -> Схлопываю T + T в S на кубите index=2
  -> Схлопываю T + T в S на кубите index=0
  -> Схлопываю T + T в S на кубите index=0
  -> Схлопываю T + T в S на кубите index=1
  -> Схлопываю T + T в S на кубите index=0
  -> Схлопываю T + T в S на кубите index=0
  -> Схлопываю T + T в S на кубите index=3
  -> Схлопываю T + T в S на кубите index=2
  -> Схлопываю T + T в S на кубите index=2
  -> Схлопываю T + T в S на кубите index=2
  -> Схлопываю T + T в S на кубите index=3
Математика Т-гейтов исправна? 🟢 ДА (Хирург чист)


In [ ]:
import random
from qiskit import QuantumCircuit
from qiskit.circuit.library import CXGate
from qiskit.quantum_info import Statevector

# ... (весь предыдущий код генерации схемы large_qc и цикл оптимизации cnot_only_qc остаются БЕЗ изменений)

print("=== ТЕСТ 1 (НЕУЯЗВИМЫЙ): ТОЛЬКО УДАЛЕНИЕ CNOT ===")
# [КОД ЦИКЛА УДАЛЕНИЯ CNOT ИЗ ПРЕДЫДУЩЕГО ШАГА]

# Физическая валидация через Statevector
# Подаем на вход суперпозицию (гейты Адамара на все кубиты), чтобы проверить все амплитуды
input_state = QuantumCircuit(num_qubits)
input_state.h(range(num_qubits))

# Симулируем эволюцию входного состояния через обе схемы
state_original = Statevector(input_state.compose(large_qc))
state_optimized = Statevector(input_state.compose(cnot_only_qc))

# Проверяем эквивалентность векторов состояний
is_physically_clean = state_original.equiv(state_optimized)

print(f"\nФизическая математика CNOT исправна? { '🟢 ДА (Разведчик полностью подтвержден!)' if is_physically_clean else '🔴 НЕТ' }")


=== ТЕСТ 1 (НЕУЯЗВИМЫЙ): ТОЛЬКО УДАЛЕНИЕ CNOT ===

Физическая математика CNOT исправна? 🟢 ДА (Разведчик полностью подтвержден!)


In [ ]:
import random
from qiskit import QuantumCircuit
from qiskit.circuit.library import CXGate, TGate, SGate
from qiskit.quantum_info import Statevector

class QuantumLumberjackOptimizer:
    """
    Промышленный двухпроходной компилятор квантовых цепей Clifford+T для Kaggle.
    Полностью валидирован по квантовым состояниям Statevector (100% стабильность).
    """
    def __init__(self, circuit: QuantumCircuit):
        self.circuit = circuit.copy()

    def prune_dead_cnots(self, source_circuit: QuantumCircuit) -> QuantumCircuit:
        """ ПРОХОД 1: Разведка и расчистка стартовых плотин CNOT. """
        num_qubits = source_circuit.num_qubits
        qubit_touched = {i: False for i in range(num_qubits)}
        clean_qc = QuantumCircuit(*source_circuit.qregs)

        for instruction in source_circuit.data:
            gate = instruction.operation
            qubit_indices = [source_circuit.find_bit(q).index for q in instruction.qubits]

            if isinstance(gate, CXGate):
                ctrl_idx = qubit_indices[0]
                tgt_idx = qubit_indices[1]

                # CNOT холостой, только если обе линии вообще ни разу не использовались
                if not qubit_touched[ctrl_idx] and not qubit_touched[tgt_idx]:
                    qubit_touched[ctrl_idx] = True
                    qubit_touched[tgt_idx] = True
                    continue

                qubit_touched[ctrl_idx] = True
                qubit_touched[tgt_idx] = True
            else:
                for q_idx in qubit_indices:
                    qubit_touched[q_idx] = True

            clean_qc.append(instruction)

        return clean_qc

    def optimize_t_ngrams(self, source_circuit: QuantumCircuit) -> QuantumCircuit:
        """ ПРОХОД 2: Хирургия и Тетрис T-гейтов по глобальному таймлайну. """
        final_qc = QuantumCircuit(*source_circuit.qregs)

        for instruction in source_circuit.data:
            gate = instruction.operation
            qubit_indices = [source_circuit.find_bit(q).index for q in instruction.qubits]

            if isinstance(gate, TGate):
                q_idx = qubit_indices[0]

                if len(final_qc.data) > 0:
                    last_instruction = final_qc.data[-1]
                    last_gate = last_instruction.operation
                    last_qubits = [final_qc.find_bit(q).index for q in last_instruction.qubits]

                    # Схлопываем T + T -> S строго встык по оригинальным объектам кубитов
                    if isinstance(last_gate, TGate) and last_qubits == [q_idx]:
                        final_qc.data.pop()
                        final_qc.append(SGate(), instruction.qubits)
                        continue

            final_qc.append(instruction)

        return final_qc

    def compile(self) -> QuantumCircuit:
        """ Запуск полного конвейера квантового сжатия. """
        # Фаза 1: Очистка CNOT плотин
        phase1_qc = self.prune_dead_cnots(self.circuit)
        # Фаза 2: Схлопывание T-биграмм
        phase2_qc = self.optimize_t_ngrams(phase1_qc)
        return phase2_qc


In [ ]:
if __name__ == "__main__":
    from qiskit import QuantumCircuit

    # 1. Создаем тестовую схему с "лишними" элементами для проверки
    # (два CNOT подряд на одних кубитах взаимно уничтожаются, а T + T = S)
    test_qc = QuantumCircuit(2)

    # Тест для метода prune_dead_cnots:
    test_qc.cx(0, 1)
    test_qc.cx(0, 1)  # Этот CNOT должен удалиться, так как кубиты не менялись

    # Тест для метода optimize_t_ngrams:
    test_qc.t(0)
    test_qc.t(0)      # T + T должны схлопнуться в один S-вентиль

    print("--- Исходная схема ---")
    print(test_qc.draw(output='text'))

    # 2. Инициализируем наш оптимизатор и запускаем компиляцию
    optimizer = QuantumLumberjackOptimizer(test_qc)
    optimized_qc = optimizer.compile()

    print("\n--- Оптимизированная схема ---")
    print(optimized_qc.draw(output='text'))

    # 3. Проверяем эквивалентность состояний до и после оптимизации
    state_init = Statevector.from_instruction(test_qc)
    state_opt = Statevector.from_instruction(optimized_qc)

    if state_init.equiv(state_opt):
        print("\n✅ Успех: Оптимизированная схема эквивалентна исходной!")
    else:
        print("\n❌ Ошибка: Логика схемы нарушена при оптимизации.")


--- Исходная схема ---
               ┌───┐┌───┐
q_0: ──■────■──┤ T ├┤ T ├
     ┌─┴─┐┌─┴─┐└───┘└───┘
q_1: ┤ X ├┤ X ├──────────
     └───┘└───┘          

--- Оптимизированная схема ---
          ┌───┐
q_0: ──■──┤ S ├
     ┌─┴─┐└───┘
q_1: ┤ X ├─────
     └───┘     

✅ Успех: Оптимизированная схема эквивалентна исходной!


In [ ]:
if __name__ == "__main__":
    import random
    from qiskit import QuantumCircuit
    from qiskit.quantum_info import Statevector

    # 1. Генерируем тестовую схему примерно из 40 вентилей
    # Используем 3 кубита для наглядности отображения в консоли
    test_qc = QuantumCircuit(3)

    print("Генерация случайной цепочки вентилей...")

    # Задаем фиксированный сид для воспроизводимости теста
    random.seed(42)

    # Наполняем схему избыточными вентилями (около 40 штук)
    for _ in range(20):
        # Добавляем случайные T-вентили, иногда подряд
        qubit = random.randint(0, 2)
        test_qc.t(qubit)
        if random.random() > 0.4:
            test_qc.t(qubit)  # Потенциальное схлопывание T + T -> S

        # Добавляем случайные CNOT, иногда дублируя их подряд
        ctrl, tgt = random.sample([0, 1, 2], 2)
        test_qc.cx(ctrl, tgt)
        if random.random() > 0.5:
            test_qc.cx(ctrl, tgt)  # Потенциальное взаимное уничтожение CNOT

    print(f"Изначально сгенерировано вентилей: {len(test_qc.data)}")

    # 2. Выводим исходную схему
    print("\n" + "="*40 + " ИСХОДНАЯ СХЕМА (40+ вентилей) " + "="*40)
    print(test_qc.draw(output='text'))

    # 3. Запускаем наш оптимизатор
    optimizer = QuantumLumberjackOptimizer(test_qc)
    optimized_qc = optimizer.compile()

    # 4. Выводим результат оптимизации
    print("\n" + "="*40 + " ОПТИМИЗИРОВАННАЯ СХЕМА " + "="*40)
    print(optimized_qc.draw(output='text'))
    print(f"Осталось вентилей после оптимизации: {len(optimized_qc.data)}")

    # 5. Математическая проверка эквивалентности через Statevector
    state_init = Statevector.from_instruction(test_qc)
    state_opt = Statevector.from_instruction(optimized_qc)

    if state_init.equiv(state_opt):
        print("\n✅ Успех: Оптимизированная схема математически эквивалентна исходной!")
    else:
        print("\n❌ Внимание: Оптимизатор изменил логику схемы. Проверьте индексы кубитов!")


Генерация случайной цепочки вентилей...
Изначально сгенерировано вентилей: 58

======================================== ИСХОДНАЯ СХЕМА (40+ вентилей) ========================================
     ┌───┐┌───┐┌───┐┌───┐┌───┐          ┌───┐┌───┐                         »
q_0: ┤ T ├┤ T ├┤ X ├┤ X ├┤ T ├──■────■──┤ T ├┤ T ├──■────■────■────────────»
     └───┘├───┤└─┬─┘└─┬─┘└───┘  │    │  ├───┤├───┤  │    │  ┌─┴─┐┌───┐     »
q_1: ─────┤ X ├──┼────┼─────────┼────┼──┤ X ├┤ T ├──┼────┼──┤ X ├┤ T ├──■──»
     ┌───┐└─┬─┘  │    │       ┌─┴─┐┌─┴─┐└─┬─┘├───┤┌─┴─┐┌─┴─┐└───┘└───┘┌─┴─┐»
q_2: ┤ T ├──■────■────■───────┤ X ├┤ X ├──■──┤ T ├┤ X ├┤ X ├──────────┤ X ├»
     └───┘                    └───┘└───┘     └───┘└───┘└───┘          └───┘»
«                                        ┌───┐┌───┐┌───┐┌───┐          ┌───┐»
«q_0: ───────■───────────────────────────┤ X ├┤ X ├┤ T ├┤ T ├──■────■──┤ X ├»
«          ┌─┴─┐┌───┐┌───┐┌───┐┌───┐     └─┬─┘└─┬─┘└───┘└───┘┌─┴─┐┌─┴─┐└─┬─┘»
«q_1: ──■──┤ X ├┤ T ├┤ T ├┤ X ├┤ X ├

In [ ]:
import random
from qiskit import QuantumCircuit
from qiskit.circuit.library import CXGate, TGate, SGate, XGate
from qiskit.quantum_info import Statevector

class QuantumLumberjackOptimizer:
    """
    Продвинутый компилятор: удаляет лишние CNOT, схлопывает T+T->S,
    а теперь еще и уничтожает парные X-вентили (X+X = уничтожение).
    """
    def __init__(self, circuit: QuantumCircuit):
        self.circuit = circuit.copy()

    def prune_dead_cnots(self, source_circuit: QuantumCircuit) -> QuantumCircuit:
        num_qubits = source_circuit.num_qubits
        qubit_touched = {i: False for i in range(num_qubits)}
        clean_qc = QuantumCircuit(*source_circuit.qregs)

        for instruction in source_circuit.data:
            gate = instruction.operation
            qubit_indices = [source_circuit.find_bit(q).index for q in instruction.qubits]

            if isinstance(gate, CXGate):
                ctrl_idx = qubit_indices[0]
                tgt_idx = qubit_indices[1]

                if not qubit_touched[ctrl_idx] and not qubit_touched[tgt_idx]:
                    qubit_touched[ctrl_idx] = True
                    qubit_touched[tgt_idx] = True
                    continue

                qubit_touched[ctrl_idx] = True
                qubit_touched[tgt_idx] = True
            else:
                for q_idx in qubit_indices:
                    qubit_touched[q_idx] = True

            clean_qc.append(instruction)

        return clean_qc

    def optimize_t_ngrams(self, source_circuit: QuantumCircuit) -> QuantumCircuit:
        final_qc = QuantumCircuit(*source_circuit.qregs)

        for instruction in source_circuit.data:
            gate = instruction.operation
            qubit_indices = [source_circuit.find_bit(q).index for q in instruction.qubits]

            if isinstance(gate, TGate):
                q_idx = qubit_indices[0]

                if len(final_qc.data) > 0:
                    last_instruction = final_qc.data[-1]
                    last_gate = last_instruction.operation
                    last_qubits = [final_qc.find_bit(q).index for q in last_instruction.qubits]

                    if isinstance(last_gate, TGate) and last_qubits == [q_idx]:
                        final_qc.data.pop()
                        final_qc.append(SGate(), instruction.qubits)
                        continue

            final_qc.append(instruction)

        return final_qc

    def optimize_x_pairs(self, source_circuit: QuantumCircuit) -> QuantumCircuit:
        """
        НОВЫЙ МЕТОД: Находит два X-вентиля подряд на одном кубите и взаимно уничтожает их.
        """
        final_qc = QuantumCircuit(*source_circuit.qregs)

        for instruction in source_circuit.data:
            gate = instruction.operation
            qubit_indices = [source_circuit.find_bit(q).index for q in instruction.qubits]

            # Если текущий вентиль — это X
            if isinstance(gate, XGate):
                q_idx = qubit_indices[0]

                # Проверяем, есть ли уже вентили в собираемой схеме
                if len(final_qc.data) > 0:
                    last_instruction = final_qc.data[-1]
                    last_gate = last_instruction.operation
                    last_qubits = [final_qc.find_bit(q).index for q in last_instruction.qubits]

                    # Если предыдущий вентиль тоже был X на этом же кубите
                    if isinstance(last_gate, XGate) and last_qubits == [q_idx]:
                        # Удаляем предыдущий X и НЕ добавляем текущий (они взаимно уничтожились!)
                        final_qc.data.pop()
                        continue

            final_qc.append(instruction)

        return final_qc

    def compile(self) -> QuantumCircuit:
        # Фаза 1: Очистка CNOT-цепочек
        phase1_qc = self.prune_dead_cnots(self.circuit)
        # Фаза 2: Схлопывание T-вентилей
        phase2_qc = self.optimize_t_ngrams(phase1_qc)
        # Фаза 3: Взаимное уничтожение X-вентилей
        phase3_qc = self.optimize_x_pairs(phase2_qc)
        return phase3_qc


if __name__ == "__main__":
    # Тестовый запуск с генерацией схем
    test_qc = QuantumCircuit(3)

    print("Генерация случайной цепочки вентилей с избыточными X...")
    random.seed(42)

    for _ in range(20):
        qubit = random.randint(0, 2)

        # Подмешиваем X-вентили (иногда парами подряд)
        test_qc.x(qubit)
        if random.random() > 0.5:
            test_qc.x(qubit)

        # Подмешиваем T-вентили
        test_qc.t(qubit)
        if random.random() > 0.4:
            test_qc.t(qubit)

        # Добавляем CNOT
        ctrl, tgt = random.sample(range(3), 2)
        test_qc.cx(ctrl, tgt)
        if random.random() > 0.5:
            test_qc.cx(ctrl, tgt)

    print(f"Изначально сгенерировано вентилей: {len(test_qc.data)}")

    print("\n" + "="*40 + " ИСХОДНАЯ СХЕМА (с X-вентилями) " + "="*40)
    print(test_qc.draw(output='text'))

    # Запускаем обновленный компилятор
    optimizer = QuantumLumberjackOptimizer(test_qc)
    optimized_qc = optimizer.compile()

    print("\n" + "="*40 + " ОПТИМИЗИРОВАННАЯ СХЕМА " + "="*40)
    print(optimized_qc.draw(output='text'))
    print(f"Осталось вентилей после оптимизации: {len(optimized_qc.data)}")

    # Проверка эквивалентности
    state_init = Statevector.from_instruction(test_qc)
    state_opt = Statevector.from_instruction(optimized_qc)

    if state_init.equiv(state_opt):
        print("\n✅ Успех: Оптимизированная схема математически эквивалентна исходной!")
    else:
        print("\n❌ Внимание: Нарушена математическая логика схем!")


Генерация случайной цепочки вентилей с избыточными X...
Изначально сгенерировано вентилей: 95

======================================== ИСХОДНАЯ СХЕМА (с X-вентилями) ========================================
                         ┌───┐┌───┐┌───┐┌───┐     ┌───┐┌───┐┌───┐          »
q_0: ─────────────────■──┤ X ├┤ X ├┤ T ├┤ T ├──■──┤ X ├┤ T ├┤ T ├──■────■──»
                      │  └───┘└───┘└───┘└───┘┌─┴─┐└───┘└───┘└───┘  │    │  »
q_1: ─────────────────┼──────────────────────┤ X ├─────────────────┼────┼──»
     ┌───┐┌───┐┌───┐┌─┴─┐                    └───┘               ┌─┴─┐┌─┴─┐»
q_2: ┤ X ├┤ T ├┤ T ├┤ X ├────────────────────────────────────────┤ X ├┤ X ├»
     └───┘└───┘└───┘└───┘                                        └───┘└───┘»
«     ┌───┐┌───┐┌───┐┌───┐┌───┐┌───┐                                        »
«q_0: ┤ X ├┤ X ├┤ T ├┤ X ├┤ X ├┤ T ├──■─────────────────────────────────────»
«     └───┘└───┘└───┘├───┤├───┤└───┘┌─┴─┐┌───┐┌───┐┌───┐┌───┐┌───┐┌───┐┌───┐»
«q_1: ─────────────

In [ ]:
from qiskit.circuit import CircuitInstruction  # Добавляем этот импорт в самый верх файла

def optimize_4_grams(self, source_circuit: QuantumCircuit) -> QuantumCircuit:
    """
    ИСПРАВЛЕННЫЙ ЭШЕЛОН: Сканирует схему скользящим окном размера 4 и схлопывает 4-граммы,
    создавая новые неизменяемые объекты CircuitInstruction.
    """
    instructions = list(source_circuit.data)
    optimized = True

    while optimized:
        optimized = False
        i = 0
        while i <= len(instructions) - 4:
            window = instructions[i:i+4]
            gates = [inst.operation for inst in window]
            qubits_per_gate = [[source_circuit.find_bit(q).index for q in inst.qubits] for inst in window]

            # --- ПРАВИЛО 1: T + T + T + T -> Z (на одном кубите) ---
            if (all(isinstance(g, TGate) for g in gates) and
                all(q == qubits_per_gate[0] for q in qubits_per_gate)):

                # Создаем новую чистую инструкцию ZGate для тех же кубитов
                z_instruction = CircuitInstruction(ZGate(), window[0].qubits, window[0].clbits)

                instructions[i:i+4] = [z_instruction]
                optimized = True
                break

            # --- ПРАВИЛО 2: X + X + X + X -> Удаление (на одном кубите) ---
            if (all(isinstance(g, XGate) for g in gates) and
                all(q == qubits_per_gate[0] for q in qubits_per_gate)):

                instructions[i:i+4] = []
                optimized = True
                break

            # --- ПРАВИЛО 3: Перенос фазы через CNOT: [CX(0,1), T(0), CX(0,1), T(0)] -> S(0) ---
            if (isinstance(gates[0], CXGate) and isinstance(gates[1], TGate) and
                isinstance(gates[2], CXGate) and isinstance(gates[3], TGate)):

                if (qubits_per_gate[0] == qubits_per_gate[2] and
                    qubits_per_gate[1] == [qubits_per_gate[0][0]] and
                    qubits_per_gate[3] == [qubits_per_gate[0][0]]):

                    # Создаем новую чистую инструкцию SGate на управляющем кубите
                    s_instruction = CircuitInstruction(SGate(), window[1].qubits, window[1].clbits)

                    instructions[i:i+4] = [s_instruction]
                    optimized = True
                    break

            i += 1

    clean_qc = QuantumCircuit(*source_circuit.qregs)
    for inst in instructions:
        clean_qc.append(inst)
    return clean_qc


In [ ]:
import random
from qiskit import QuantumCircuit
from qiskit.circuit import CircuitInstruction  # Корректный импорт для создания инструкций
from qiskit.circuit.library import CXGate, TGate, SGate, XGate, ZGate
from qiskit.quantum_info import Statevector

class QuantumLumberjackOptimizer:
    """
    Продвинутый компилятор: удаляет лишние CNOT, схлопывает T+T->S,
    уничтожает парные X-вентили, а теперь еще и анализирует 4-граммы.
    """
    def __init__(self, circuit: QuantumCircuit):
        self.circuit = circuit.copy()

    def prune_dead_cnots(self, source_circuit: QuantumCircuit) -> QuantumCircuit:
        num_qubits = source_circuit.num_qubits
        qubit_touched = {i: False for i in range(num_qubits)}
        clean_qc = QuantumCircuit(*source_circuit.qregs)

        for instruction in source_circuit.data:
            gate = instruction.operation
            qubit_indices = [source_circuit.find_bit(q).index for q in instruction.qubits]

            if isinstance(gate, CXGate):
                ctrl_idx = qubit_indices[0]
                tgt_idx = qubit_indices[1]

                if not qubit_touched[ctrl_idx] and not qubit_touched[tgt_idx]:
                    qubit_touched[ctrl_idx] = True
                    qubit_touched[tgt_idx] = True
                    continue

                qubit_touched[ctrl_idx] = True
                qubit_touched[tgt_idx] = True
            else:
                for q_idx in qubit_indices:
                    qubit_touched[q_idx] = True

            clean_qc.append(instruction)
        return clean_qc

    def optimize_t_ngrams(self, source_circuit: QuantumCircuit) -> QuantumCircuit:
        final_qc = QuantumCircuit(*source_circuit.qregs)
        for instruction in source_circuit.data:
            gate = instruction.operation
            qubit_indices = [source_circuit.find_bit(q).index for q in instruction.qubits]

            if isinstance(gate, TGate):
                q_idx = qubit_indices[0]
                if len(final_qc.data) > 0:
                    last_instruction = final_qc.data[-1]
                    last_gate = last_instruction.operation
                    last_qubits = [final_qc.find_bit(q).index for q in last_instruction.qubits]

                    if isinstance(last_gate, TGate) and last_qubits == [q_idx]:
                        final_qc.data.pop()
                        final_qc.append(SGate(), instruction.qubits)
                        continue
            final_qc.append(instruction)
        return final_qc

    def optimize_x_pairs(self, source_circuit: QuantumCircuit) -> QuantumCircuit:
        final_qc = QuantumCircuit(*source_circuit.qregs)
        for instruction in source_circuit.data:
            gate = instruction.operation
            qubit_indices = [source_circuit.find_bit(q).index for q in instruction.qubits]

            if isinstance(gate, XGate):
                q_idx = qubit_indices[0]
                if len(final_qc.data) > 0:
                    last_instruction = final_qc.data[-1]
                    last_gate = last_instruction.operation
                    last_qubits = [final_qc.find_bit(q).index for q in last_instruction.qubits]

                    if isinstance(last_gate, XGate) and last_qubits == [q_idx]:
                        final_qc.data.pop()
                        continue
            final_qc.append(instruction)
        return final_qc

    def optimize_4_grams(self, source_circuit: QuantumCircuit) -> QuantumCircuit:
        """
        ИСПРАВЛЕННЫЙ ЭШЕЛОН: Сканирует схему скользящим окном размера 4 и схлопывает 4-граммы,
        исправляя ошибки извлечения qubits/clbits из window.
        """
        instructions = list(source_circuit.data)
        optimized = True

        while optimized:
            optimized = False
            i = 0
            while i <= len(instructions) - 4:
                window = instructions[i:i+4]
                gates = [inst.operation for inst in window]
                qubits_per_gate = [[source_circuit.find_bit(q).index for q in inst.qubits] for inst in window]

                # --- ПРАВИЛО 1: T + T + T + T -> Z (на одном кубите) ---
                if (all(isinstance(g, TGate) for g in gates) and
                    all(q == qubits_per_gate[0] for q in qubits_per_gate)):

                    # Берем кубиты и классические биты из первой инструкции окна
                    z_instruction = CircuitInstruction(ZGate(), window[0].qubits, window[0].clbits)

                    instructions[i:i+4] = [z_instruction]
                    optimized = True
                    break

                # --- ПРАВИЛО 2: X + X + X + X -> Полное удаление ---
                if (all(isinstance(g, XGate) for g in gates) and
                    all(q == qubits_per_gate[0] for q in qubits_per_gate)):

                    instructions[i:i+4] = []
                    optimized = True
                    break

                # --- ПРАВИЛО 3: Перенос фазы через CNOT: [CX(0,1), T(0), CX(0,1), T(0)] -> S(0) ---
                if (isinstance(gates[0], CXGate) and isinstance(gates[1], TGate) and
                    isinstance(gates[2], CXGate) and isinstance(gates[3], TGate)):

                    # Условие: CNOT-ы идентичны, а T сидят на управляющих кубитах (qubits_per_gate[0][0])
                    if (qubits_per_gate[0] == qubits_per_gate[2] and
                        qubits_per_gate[1] == [qubits_per_gate[0][0]] and
                        qubits_per_gate[3] == [qubits_per_gate[0][0]]):

                        # Превращаем каскад в один SGate на управляющем кубите (кубит взят из T-вентиля)
                        s_instruction = CircuitInstruction(SGate(), window[1].qubits, window[1].clbits)

                        instructions[i:i+4] = [s_instruction]
                        optimized = True
                        break

                i += 1

        clean_qc = QuantumCircuit(*source_circuit.qregs)
        for inst in instructions:
            clean_qc.append(inst)
        return clean_qc

    def compile(self) -> QuantumCircuit:
        phase1_qc = self.prune_dead_cnots(self.circuit)
        phase2_qc = self.optimize_t_ngrams(phase1_qc)
        phase3_qc = self.optimize_x_pairs(phase2_qc)
        phase4_qc = self.optimize_4_grams(phase3_qc)
        return phase4_qc


if __name__ == "__main__":
    # Тестовый стенд для проверки 4-грамм
    test_qc = QuantumCircuit(2)

    # Добавляем 4-грамму T+T+T+T на кубит 0
    for _ in range(4): test_qc.t(0)

    # Добавляем структуру [CX, T, CX, T]
    test_qc.cx(0, 1)
    test_qc.t(0)
    test_qc.cx(0, 1)
    test_qc.t(0)

    # Добавляем 4-грамму X+X+X+X на кубит 1
    for _ in range(4): test_qc.x(1)

    print("="*40 + " ИСХОДНАЯ ТЕСТОВАЯ СХЕМА " + "="*40)
    print(test_qc.draw(output='text'))

    # Запуск компиляции
    optimizer = QuantumLumberjackOptimizer(test_qc)
    optimized_qc = optimizer.compile()

    print("\n" + "="*40 + " ПОСЛЕ 4-ЭТАПНОЙ ОПТИМИЗАЦИИ " + "="*40)
    print(optimized_qc.draw(output='text'))

    # Математическая верификация
    state_init = Statevector.from_instruction(test_qc)
    state_opt = Statevector.from_instruction(optimized_qc)

    if state_init.equiv(state_opt):
        print("\n✅ Успех: 4-граммы успешно оптимизированы, эквивалентность сохранена!")
    else:
        print("\n❌ Ошибка: Логика 4-грамм нарушила эквивалентность!")


======================================== ИСХОДНАЯ ТЕСТОВАЯ СХЕМА ========================================
     ┌───┐┌───┐┌───┐┌───┐     ┌───┐     ┌───┐               
q_0: ┤ T ├┤ T ├┤ T ├┤ T ├──■──┤ T ├──■──┤ T ├───────────────
     └───┘└───┘└───┘└───┘┌─┴─┐└───┘┌─┴─┐├───┤┌───┐┌───┐┌───┐
q_1: ────────────────────┤ X ├─────┤ X ├┤ X ├┤ X ├┤ X ├┤ X ├
                         └───┘     └───┘└───┘└───┘└───┘└───┘

======================================== ПОСЛЕ 4-ЭТАПНОЙ ОПТИМИЗАЦИИ ========================================
     ┌───┐┌───┐┌───┐
q_0: ┤ S ├┤ S ├┤ S ├
     └───┘└───┘└───┘
q_1: ───────────────
                    

✅ Успех: 4-граммы успешно оптимизированы, эквивалентность сохранена!


In [ ]:
import random
from qiskit import QuantumCircuit
from qiskit.circuit import CircuitInstruction
from qiskit.circuit.library import CXGate, TGate, SGate, XGate, ZGate
from qiskit.quantum_info import Statevector

class QuantumLumberjackOptimizer:
    """
    Итеративный компилятор: прогоняет 4 фазы оптимизации по кругу
    до тех пор, пока схема не перестанет уменьшаться.
    """
    def __init__(self, circuit: QuantumCircuit):
        self.circuit = circuit.copy()

    def prune_dead_cnots(self, source_circuit: QuantumCircuit) -> QuantumCircuit:
        num_qubits = source_circuit.num_qubits
        qubit_touched = {i: False for i in range(num_qubits)}
        clean_qc = QuantumCircuit(*source_circuit.qregs)

        for instruction in source_circuit.data:
            gate = instruction.operation
            qubit_indices = [source_circuit.find_bit(q).index for q in instruction.qubits]

            if isinstance(gate, CXGate):
                ctrl_idx = qubit_indices[0]
                tgt_idx = qubit_indices[1]

                if not qubit_touched[ctrl_idx] and not qubit_touched[tgt_idx]:
                    qubit_touched[ctrl_idx] = True
                    qubit_touched[tgt_idx] = True
                    continue

                qubit_touched[ctrl_idx] = True
                qubit_touched[tgt_idx] = True
            else:
                for q_idx in qubit_indices:
                    qubit_touched[q_idx] = True

            clean_qc.append(instruction)
        return clean_qc

    def optimize_t_ngrams(self, source_circuit: QuantumCircuit) -> QuantumCircuit:
        final_qc = QuantumCircuit(*source_circuit.qregs)
        for instruction in source_circuit.data:
            gate = instruction.operation
            qubit_indices = [source_circuit.find_bit(q).index for q in instruction.qubits]

            if isinstance(gate, TGate):
                q_idx = qubit_indices[0]
                if len(final_qc.data) > 0:
                    last_instruction = final_qc.data[-1]
                    last_gate = last_instruction.operation
                    last_qubits = [final_qc.find_bit(q).index for q in last_instruction.qubits]

                    if isinstance(last_gate, TGate) and last_qubits == [q_idx]:
                        final_qc.data.pop()
                        final_qc.append(SGate(), instruction.qubits)
                        continue
            final_qc.append(instruction)
        return final_qc

    def optimize_x_pairs(self, source_circuit: QuantumCircuit) -> QuantumCircuit:
        final_qc = QuantumCircuit(*source_circuit.qregs)
        for instruction in source_circuit.data:
            gate = instruction.operation
            qubit_indices = [source_circuit.find_bit(q).index for q in instruction.qubits]

            if isinstance(gate, XGate):
                q_idx = qubit_indices[0]
                if len(final_qc.data) > 0:
                    last_instruction = final_qc.data[-1]
                    last_gate = last_instruction.operation
                    last_qubits = [final_qc.find_bit(q).index for q in last_instruction.qubits]

                    if isinstance(last_gate, XGate) and last_qubits == [q_idx]:
                        final_qc.data.pop()
                        continue
            final_qc.append(instruction)
        return final_qc

    def optimize_4_grams(self, source_circuit: QuantumCircuit) -> QuantumCircuit:
        instructions = list(source_circuit.data)
        optimized = True

        while optimized:
            optimized = False
            i = 0
            while i <= len(instructions) - 4:
                window = instructions[i:i+4]
                gates = [inst.operation for inst in window]
                qubits_per_gate = [[source_circuit.find_bit(q).index for q in inst.qubits] for inst in window]

                # --- ПРАВИЛО 1: T + T + T + T -> Z ---
                if (all(isinstance(g, TGate) for g in gates) and
                    all(q == qubits_per_gate[0] for q in qubits_per_gate)):

                    z_instruction = CircuitInstruction(ZGate(), window[0].qubits, window[0].clbits)
                    instructions[i:i+4] = [z_instruction]
                    optimized = True
                    break

                # --- ПРАВИЛО 2: X + X + X + X -> Удаление ---
                if (all(isinstance(g, XGate) for g in gates) and
                    all(q == qubits_per_gate[0] for q in qubits_per_gate)):

                    instructions[i:i+4] = []
                    optimized = True
                    break

                # --- ПРАВИЛО 3: Перенос фазы через CNOT: [CX, T, CX, T] -> S ---
                if (isinstance(gates[0], CXGate) and isinstance(gates[1], TGate) and
                    isinstance(gates[2], CXGate) and isinstance(gates[3], TGate)):

                    if (qubits_per_gate[0] == qubits_per_gate[2] and
                        qubits_per_gate[1] == [qubits_per_gate[0][0]] and
                        qubits_per_gate[3] == [qubits_per_gate[0][0]]):

                        s_instruction = CircuitInstruction(SGate(), window[1].qubits, window[1].clbits)
                        instructions[i:i+4] = [s_instruction]
                        optimized = True
                        break

                i += 1

        clean_qc = QuantumCircuit(*source_circuit.qregs)
        for inst in instructions:
            clean_qc.append(inst)
        return clean_qc

    def optimize_s_pairs(self, source_circuit: QuantumCircuit) -> QuantumCircuit:
        """
        ДОПОЛНИТЕЛЬНЫЙ БОНУС: схлопывание S + S -> Z на одном кубите.
        Поможет собрать остатки после многопроходного цикла!
        """
        final_qc = QuantumCircuit(*source_circuit.qregs)
        for instruction in source_circuit.data:
            gate = instruction.operation
            qubit_indices = [source_circuit.find_bit(q).index for q in instruction.qubits]

            if isinstance(gate, SGate):
                q_idx = qubit_indices[0]
                if len(final_qc.data) > 0:
                    last_instruction = final_qc.data[-1]
                    last_gate = last_instruction.operation
                    last_qubits = [final_qc.find_bit(q).index for q in last_instruction.qubits]

                    if isinstance(last_gate, SGate) and last_qubits == [q_idx]:
                        final_qc.data.pop()
                        final_qc.append(ZGate(), instruction.qubits)
                        continue
            final_qc.append(instruction)
        return final_qc

    def compile(self) -> QuantumCircuit:
        """
        ГЛАВНЫЙ ИТЕРАЦИОННЫЙ ЦИКЛ (Конвейер)
        """
        current_qc = self.circuit.copy()

        # Будем крутить цикл, пока количество вентилей уменьшается
        previous_size = len(current_qc.data) + 1
        iteration = 1

        print(f"\nЗапуск итеративной компиляции...")
        while len(current_qc.data) < previous_size:
            previous_size = len(current_qc.data)

            # Последовательный запуск фаз оптимизации
            current_qc = self.prune_dead_cnots(current_qc)
            current_qc = self.optimize_t_ngrams(current_qc)
            current_qc = self.optimize_x_pairs(current_qc)
            current_qc = self.optimize_4_grams(current_qc)
            current_qc = self.optimize_s_pairs(current_qc)  # Проверяем S-пары

            # Если размер изменился, логируем проход
            if len(current_qc.data) < previous_size:
                print(f" -> Проход {iteration}: вентилей осталось {len(current_qc.data)}")
                iteration += 1

        return current_qc


if __name__ == "__main__":
    # Воссоздаем в точности схему с предыдущего шага
    test_qc = QuantumCircuit(2)

    # [T, T, T, T, CX, T, CX, T] на кубите 0 и CNOT на кубит 1
    for _ in range(4): test_qc.t(0)
    test_qc.cx(0, 1)
    test_qc.t(0)
    test_qc.cx(0, 1)
    test_qc.t(0)

    # Блок из четырех X на кубите 1
    for _ in range(4): test_qc.x(1)

    print("="*40 + " ИСХОДНАЯ СХЕМА С ПРЕДЫДУЩЕГО ШАГА " + "="*40)
    print(test_qc.draw(output='text'))

    # Запуск итерационного компилятора
    optimizer = QuantumLumberjackOptimizer(test_qc)
    optimized_qc = optimizer.compile()

    print("\n" + "="*40 + " ФИНАЛЬНЫЙ РЕЗУЛЬТАТ ПОСЛЕ ВСЕХ ЦИКЛОВ " + "="*40)
    print(optimized_qc.draw(output='text'))

    # Математическая верификация
    state_init = Statevector.from_instruction(test_qc)
    state_opt = Statevector.from_instruction(optimized_qc)

    if state_init.equiv(state_opt):
        print("\n✅ Успех: Циклическая оптимизация завершена, эквивалентность идеальна!")
    else:
        print("\n❌ Ошибка: Нарушена эквивалентность при циклической обработке!")


======================================== ИСХОДНАЯ СХЕМА С ПРЕДЫДУЩЕГО ШАГА ========================================
     ┌───┐┌───┐┌───┐┌───┐     ┌───┐     ┌───┐               
q_0: ┤ T ├┤ T ├┤ T ├┤ T ├──■──┤ T ├──■──┤ T ├───────────────
     └───┘└───┘└───┘└───┘┌─┴─┐└───┘┌─┴─┐├───┤┌───┐┌───┐┌───┐
q_1: ────────────────────┤ X ├─────┤ X ├┤ X ├┤ X ├┤ X ├┤ X ├
                         └───┘     └───┘└───┘└───┘└───┘└───┘

Запуск итеративной компиляции...
 -> Проход 1: вентилей осталось 2

======================================== ФИНАЛЬНЫЙ РЕЗУЛЬТАТ ПОСЛЕ ВСЕХ ЦИКЛОВ ========================================
     ┌───┐┌───┐
q_0: ┤ Z ├┤ S ├
     └───┘└───┘
q_1: ──────────
               

✅ Успех: Циклическая оптимизация завершена, эквивалентность идеальна!


In [ ]:
if __name__ == "__main__":
    # Тестовый стенд для большой случайной схемы
    test_qc = QuantumCircuit(3)

    print("Генерация большой случайной цепочки вентилей...")
    random.seed(42)  # Тот же сид для воспроизводимости сложных структур

    # Генерируем массивную схему (около 90+ вентилей)
    for _ in range(25):
        qubit = random.randint(0, 2)

        # Подмешиваем X-вентили
        test_qc.x(qubit)
        if random.random() > 0.4:
            test_qc.x(qubit)

        # Подмешиваем T-вентили
        test_qc.t(qubit)
        if random.random() > 0.3:
            test_qc.t(qubit)

        # Навешиваем CNOT-ы
        ctrl, tgt = random.sample(range(3), 2)
        test_qc.cx(ctrl, tgt)
        if random.random() > 0.4:
            test_qc.cx(ctrl, tgt)

    print(f"Изначально сгенерировано вентилей: {len(test_qc.data)}")

    print("\n" + "="*40 + " ИСХОДНАЯ МАССИВНАЯ СХЕМА " + "="*40)
    print(test_qc.draw(output='text'))

    # Запуск нашего итерационного компилятора
    optimizer = QuantumLumberjackOptimizer(test_qc)
    optimized_qc = optimizer.compile()

    print("\n" + "="*40 + " ОПТИМИЗИРОВАННАЯ СХЕМА (ФИНАЛ) " + "="*40)
    print(optimized_qc.draw(output='text'))
    print(f"Осталось вентилей после оптимизации: {len(optimized_qc.data)}")

    # Финальная математическая верификация
    state_init = Statevector.from_instruction(test_qc)
    state_opt = Statevector.from_instruction(optimized_qc)

    if state_init.equiv(state_opt):
        print("\n✅ УСПЕХ: Большая схема полностью оптимизирована! Эквивалентность сохранена.")
    else:
        print("\n❌ ВНИМАНИЕ: На большой схеме нарушена квантовая эквивалентность!")


Генерация большой случайной цепочки вентилей...
Изначально сгенерировано вентилей: 125

======================================== ИСХОДНАЯ МАССИВНАЯ СХЕМА ========================================
                         ┌───┐┌───┐┌───┐┌───┐     ┌───┐┌───┐┌───┐          »
q_0: ─────────────────■──┤ X ├┤ X ├┤ T ├┤ T ├──■──┤ X ├┤ T ├┤ T ├──■────■──»
                      │  └───┘└───┘└───┘└───┘┌─┴─┐└───┘└───┘└───┘  │    │  »
q_1: ─────────────────┼──────────────────────┤ X ├─────────────────┼────┼──»
     ┌───┐┌───┐┌───┐┌─┴─┐                    └───┘               ┌─┴─┐┌─┴─┐»
q_2: ┤ X ├┤ T ├┤ T ├┤ X ├────────────────────────────────────────┤ X ├┤ X ├»
     └───┘└───┘└───┘└───┘                                        └───┘└───┘»
«     ┌───┐┌───┐┌───┐┌───┐┌───┐┌───┐┌───┐                                   »
«q_0: ┤ X ├┤ X ├┤ T ├┤ X ├┤ X ├┤ T ├┤ T ├──■────────────────────────────────»
«     └───┘└───┘└───┘├───┤├───┤└───┘└───┘┌─┴─┐┌───┐┌───┐┌───┐┌───┐┌───┐┌───┐»
«q_1: ───────────────┤ X ├┤ X ├─

In [ ]:
!pip install pylatexenc

In [ ]:
import pkg_resources

pkg_resources.working_set.add_entry("/usr/local/lib/python3.13/dist-packages")


/tmp/ipykernel_2831/1089040168.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [ ]:
    # Исправленное автоматическое сохранение схем в текстовые файлы
    with open("initial_circuit.txt", "w", encoding="utf-8") as f:
        f.write(str(test_qc.draw(output='text')))  # Оборачиваем в str()

    with open("optimized_circuit.txt", "w", encoding="utf-8") as f:
        f.write(str(optimized_qc.draw(output='text')))  # Оборачиваем в str()

    print("\n💾 Обе схемы успешно сохранены в файлы 'initial_circuit.txt' и 'optimized_circuit.txt'!")



💾 Обе схемы успешно сохранены в файлы 'initial_circuit.txt' и 'optimized_circuit.txt'!


In [ ]:
!pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 59.7 MB/s  0:00:00


In [ ]:
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas


def save_circuits_to_pdf(filename, initial_text, optimized_text):
    c = canvas.Canvas(filename, pagesize=letter)
    width, height = letter

    # Настраиваем моноширинный шрифт (важно для текстовой графики схем)
    c.setFont("Courier", 10)

    # Стартовая позиция по вертикали
    y = height - 40

    def write_text(text_block, current_y):
        for line in text_block.split("\n"):
            if current_y < 40:  # Если страница закончилась, создаем новую
                c.showPage()
                c.setFont("Courier", 10)
                current_y = height - 40
            c.drawString(40, current_y, line)
            current_y -= 12  # Шаг строки
        return current_y

    # Пишем заголовок и первую схему
    c.drawString(40, y, "=== INITIAL CIRCUIT ===")
    y = write_text(initial_text, y - 20)

    # Пишем вторую схему
    y -= 20
    c.drawString(40, y, "=== OPTIMIZED CIRCUIT ===")
    y = write_text(optimized_text, y - 20)

    c.save()


# --- Интеграция в ваш код ---
# Вместо записи в .txt файлы передаем строки в функцию:
initial_circuit_str = str(test_qc.draw(output="text"))
optimized_circuit_str = str(optimized_qc.draw(output="text"))

save_circuits_to_pdf(
    "circuits.pdf", initial_circuit_str, optimized_circuit_str
)
print("\n💾 Обе схемы успешно сохранены в файл 'circuits.pdf'!")



💾 Обе схемы успешно сохранены в файл 'circuits.pdf'!


In [ ]:
# Сохранение начальной схемы в PNG
test_qc.draw(output="mpl", filename="initial_circuit.png")

# Сохранение оптимизированной схемы в PNG
optimized_qc.draw(output="mpl", filename="optimized_circuit.png")

print(
    "\n🎨 Обе схемы успешно сохранены как картинки 'initial_circuit.png' и 'optimized_circuit.png'!"
)


MissingOptionalLibraryError: "The 'pylatexenc' library is required to use 'MatplotlibDrawer'. You can install it with 'pip install pylatexenc'."

In [ ]:
# Сохранение начальной схемы напрямую в PDF
test_qc.draw(output="mpl", filename="initial_circuit.pdf")

# Сохранение оптимизированной схемы напрямую в PDF
optimized_qc.draw(output="mpl", filename="optimized_circuit.pdf")


MissingOptionalLibraryError: "The 'pylatexenc' library is required to use 'MatplotlibDrawer'. You can install it with 'pip install pylatexenc'."